# Introduction to Rigorous Policy Comparison for General Performance Metrics

### Load Packages for Binary Metrics (sequentialized_barnard_tests)

In [1]:
import numpy as np 
from sequentialized_barnard_tests.step import StepTest
from sequentialized_barnard_tests.savi import SaviTest
from sequentialized_barnard_tests.base import Hypothesis, Decision
import os 
import sys
from tqdm import tqdm 

### Format pathing to include parent directory (for loading NSCORE)

In [2]:
path_for_loading_nscore = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(path_for_loading_nscore)

### Load the NSCORE modules

In [3]:
# WSR METHOD: Unstructured nonparametric method (more general, but less sample efficient)
from nscore.wsr import WsrComparisonTest

# Theta-SAVI METHOD: Explicitly structured parametric method (less general, but more sample efficient)
from nscore.savi import PartialCreditSaviTest

# Our method (NSCORE): as general as WSR and as efficient as Theta-SAVI 
from nscore.nsm import BernoulliNsmTest
from nscore.nonparametric_nsm import ContinuousNsmTest

## Begin Evaluation Demonstrations

### Binary Metrics

In [4]:
# Fix seed for reproducibility
np.random.seed(42)

# Create Bernoulli data corresponding to different policies
n_policies = 2

# Construct well-spaced policy means
policy_means = 0.5 + 0.5 * ((np.arange(n_policies) - 0.5) / n_policies)

# Generate data
n_trials = 500
bernoulli_data = np.zeros((n_trials, n_policies))

for i in range(n_policies):
    bernoulli_data[:, i] = np.random.binomial(1, policy_means[i], size=n_trials)



In [5]:
# Set parameters of the tests
alpha = 0.05 

step_test = StepTest(Hypothesis.P0LessThanP1, n_trials, alpha)

savi_test = SaviTest(Hypothesis.P0LessThanP1, alpha)
pc_savi_test =  PartialCreditSaviTest(Hypothesis.P0LessThanP1, alpha, c=np.arange(2)/1.)

binary_nscore_test = BernoulliNsmTest(Hypothesis.P0LessThanP1, alpha, c=np.arange(2)/1.)
nscore_test = ContinuousNsmTest(Hypothesis.P0LessThanP1, alpha, c=np.arange(11)/10.)

wsr_test = WsrComparisonTest(Hypothesis.P0LessThanP1, alpha, c_wsr=0.95)


In [6]:
step_result = step_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])

savi_result = savi_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
pc_savi_result = pc_savi_test.run_on_sequence(bernoulli_data[:, 0].astype(int), bernoulli_data[:, 1].astype(int))

binary_nscore_result = binary_nscore_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
nscore_result = nscore_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])

wsr_result = wsr_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])


/home/dsnyder5/Documents/Github/nscore/nscore/savi.py:558: FutureWarning: Some rows of `p` do not sum to 1.0 within tolerance of eps=np.float64(2.220446049250313e-15). Currently, the last element of these rows is adjusted to compensate, but this condition will produce NaNs beginning in SciPy 1.18.0. Please ensure that rows of `p` sum to 1.0 to avoid futher disruption.
  - multinomial.logpmf(vec_datum_0, n=1, p=optimal_p_0)
/home/dsnyder5/Documents/Github/nscore/nscore/savi.py:559: FutureWarning: Some rows of `p` do not sum to 1.0 within tolerance of eps=np.float64(2.220446049250313e-15). Currently, the last element of these rows is adjusted to compensate, but this condition will produce NaNs beginning in SciPy 1.18.0. Please ensure that rows of `p` sum to 1.0 to avoid futher disruption.
  - multinomial.logpmf(vec_datum_1, n=1, p=optimal_p_1)


In [7]:
print("Binary Metric Comparison Problem")
print()
print(f"True policy mean performance levels: Policy 0 = {policy_means[0]}, Policy 1 = {policy_means[1]}")
print()
print(f"Times to decision for each method (alpha = {alpha}): ")
print()
print("STEP: ", step_result.info["Time"])
print("SAVI: ", savi_result.info["Time"])
print("Theta-SAVI: ", pc_savi_result.info["Time"])
print("Binary NSCORE: ", binary_nscore_result.info["Time"])
print("NSCORE: ", nscore_result.info["Time"])
print("WSR: ", wsr_result.info["Time"])


Binary Metric Comparison Problem

True policy mean performance levels: Policy 0 = 0.375, Policy 1 = 0.625

Times to decision for each method (alpha = 0.05): 

STEP:  65
SAVI:  65
Theta-SAVI:  64
Binary NSCORE:  62
NSCORE:  62
WSR:  144


### Initial Conclusion

We see that all methods do about the same here, with the exception of WSR, which struggles due to the high variance of the problem (equivalently, due to the low signal-to-noise ratio). 

### Were We Just Lucky? 

To test this, we need to re-run the same setting over many independent redraws of data. We do this below in order to try to get a better sense of which methods are strongest

In [8]:
n_reruns = 20

times_to_decision_per_policy = np.zeros((n_reruns, 6))

for i in tqdm(range(n_reruns)):
    # Reset the tests
    step_test.reset()
    savi_test.reset()
    pc_savi_test.reset()
    binary_nscore_test.reset()
    nscore_test.reset()
    wsr_test.reset()

    # Draw new data
    for j in range(n_policies):
        bernoulli_data[:, j] = np.random.binomial(1, policy_means[j], size=n_trials)
    
    # Run each test
    step_result = step_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    savi_result = savi_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    pc_savi_result = pc_savi_test.run_on_sequence(bernoulli_data[:, 0].astype(int), bernoulli_data[:, 1].astype(int))
    binary_nscore_result = binary_nscore_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    nscore_result = nscore_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    wsr_result = wsr_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])

    # Store the times-to-decision for each method
    times_to_decision_per_policy[i, 0] = step_result.info["Time"]
    times_to_decision_per_policy[i, 1] = savi_result.info["Time"]
    times_to_decision_per_policy[i, 2] = pc_savi_result.info["Time"]
    times_to_decision_per_policy[i, 3] = binary_nscore_result.info["Time"]
    times_to_decision_per_policy[i, 4] = nscore_result.info["Time"]
    times_to_decision_per_policy[i, 5] = wsr_result.info["Time"]



100%|██████████| 20/20 [00:56<00:00,  2.80s/it]


In [9]:
# Compute average times to decision for each method over n_reruns independent redraws
mean_time_to_decision_per_policy = np.mean(times_to_decision_per_policy, axis=0)

# Compute std of times to decision for each method over n_reruns independent redraws
std_time_to_decision_per_policy = np.std(times_to_decision_per_policy, axis=0)

print("Binary Metric Comparison Problem")
print()
print(f"True policy mean performance levels: Policy 0 = {policy_means[0]}, Policy 1 = {policy_means[1]}")
print()
print(f"Average (over {n_reruns} redraws) times to decision for each method (alpha = {alpha}): ")
print()
print(f"STEP: {mean_time_to_decision_per_policy[0]} ({std_time_to_decision_per_policy[0] / np.sqrt(n_reruns)})")
print(f"SAVI: {mean_time_to_decision_per_policy[1]} ({std_time_to_decision_per_policy[1] / np.sqrt(n_reruns)})")
print(f"Theta-SAVI: {mean_time_to_decision_per_policy[2]} ({std_time_to_decision_per_policy[2] / np.sqrt(n_reruns)})")
print(f"Binary NSCORE: {mean_time_to_decision_per_policy[3]} ({std_time_to_decision_per_policy[3] / np.sqrt(n_reruns)})")
print(f"NSCORE: {mean_time_to_decision_per_policy[4]} ({std_time_to_decision_per_policy[4] / np.sqrt(n_reruns)})")
print(f"WSR: {mean_time_to_decision_per_policy[5]} ({std_time_to_decision_per_policy[5] / np.sqrt(n_reruns)})")

Binary Metric Comparison Problem

True policy mean performance levels: Policy 0 = 0.375, Policy 1 = 0.625

Average (over 20 redraws) times to decision for each method (alpha = 0.05): 

STEP: 73.55 (11.597731459212184)
SAVI: 82.35 (15.39078864126202)
Theta-SAVI: 77.35 (15.039992519944947)
Binary NSCORE: 80.1 (15.894165596218004)
NSCORE: 84.1 (15.502564304011127)
WSR: 170.8 (33.60510377903928)


### Reminder: STEP is uniformly strong in skewed cases and small-gap cases

In [10]:
# Construct skewed means
policy_means_skewed = np.array([0.83, 0.95])

times_to_decision_per_policy_skewed = np.zeros((n_reruns, 6))

for i in tqdm(range(n_reruns)):
    # Reset the tests
    step_test.reset()
    savi_test.reset()
    pc_savi_test.reset()
    binary_nscore_test.reset()
    nscore_test.reset()
    wsr_test.reset()

    # Draw new data
    for j in range(n_policies):
        bernoulli_data[:, j] = np.random.binomial(1, policy_means_skewed[j], size=n_trials)
    
    # Run each test
    step_result = step_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    savi_result = savi_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    pc_savi_result = pc_savi_test.run_on_sequence(bernoulli_data[:, 0].astype(int), bernoulli_data[:, 1].astype(int))
    binary_nscore_result = binary_nscore_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    nscore_result = nscore_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    wsr_result = wsr_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])

    # Store the times-to-decision for each method
    times_to_decision_per_policy_skewed[i, 0] = step_result.info["Time"]
    times_to_decision_per_policy_skewed[i, 1] = savi_result.info["Time"]
    times_to_decision_per_policy_skewed[i, 2] = pc_savi_result.info["Time"]
    times_to_decision_per_policy_skewed[i, 3] = binary_nscore_result.info["Time"]
    times_to_decision_per_policy_skewed[i, 4] = nscore_result.info["Time"]
    times_to_decision_per_policy_skewed[i, 5] = wsr_result.info["Time"]

100%|██████████| 20/20 [01:04<00:00,  3.24s/it]


In [11]:
# Compute average times to decision for each method over n_reruns independent redraws
mean_time_to_decision_per_policy_skewed = np.mean(times_to_decision_per_policy_skewed, axis=0)

# Compute std of times to decision for each method over n_reruns independent redraws
std_time_to_decision_per_policy_skewed = np.std(times_to_decision_per_policy_skewed, axis=0)

print("(Skewed) Binary Metric Comparison Problem")
print()
print(f"True policy mean performance levels: Policy 0 = {policy_means_skewed[0]}, Policy 1 = {policy_means_skewed[1]}")
print()
print(f"Average (over {n_reruns} redraws) times to decision for each method (alpha = {alpha}): ")
print()
print(f"STEP: {mean_time_to_decision_per_policy_skewed[0]} ({std_time_to_decision_per_policy_skewed[0] / np.sqrt(n_reruns)})")
print(f"SAVI: {mean_time_to_decision_per_policy_skewed[1]} ({std_time_to_decision_per_policy_skewed[1] / np.sqrt(n_reruns)})")
print(f"Theta-SAVI: {mean_time_to_decision_per_policy_skewed[2]} ({std_time_to_decision_per_policy_skewed[2] / np.sqrt(n_reruns)})")
print(f"Binary NSCORE: {mean_time_to_decision_per_policy_skewed[3]} ({std_time_to_decision_per_policy_skewed[3] / np.sqrt(n_reruns)})")
print(f"NSCORE: {mean_time_to_decision_per_policy_skewed[4]} ({std_time_to_decision_per_policy_skewed[4] / np.sqrt(n_reruns)})")
print(f"WSR: {mean_time_to_decision_per_policy_skewed[5]} ({std_time_to_decision_per_policy_skewed[5] / np.sqrt(n_reruns)})")

(Skewed) Binary Metric Comparison Problem

True policy mean performance levels: Policy 0 = 0.83, Policy 1 = 0.95

Average (over 20 redraws) times to decision for each method (alpha = 0.05): 

STEP: 106.8 (12.11664144885042)
SAVI: 105.9 (16.638194012572402)
Theta-SAVI: 115.0 (16.97306690023933)
Binary NSCORE: 100.8 (17.203720527839316)
NSCORE: 102.05 (16.995069137841124)
WSR: 174.4 (32.038133528656125)


In [12]:
# Construct close means
policy_means_close = np.array([0.45, 0.55])

times_to_decision_per_policy_close = np.zeros((n_reruns, 6))

for i in tqdm(range(n_reruns)):
    # Reset the tests
    step_test.reset()
    savi_test.reset()
    pc_savi_test.reset()
    binary_nscore_test.reset()
    nscore_test.reset()
    wsr_test.reset()

    # Draw new data
    for j in range(n_policies):
        bernoulli_data[:, j] = np.random.binomial(1, policy_means_close[j], size=n_trials)
    
    # Run each test
    step_result = step_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    savi_result = savi_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    pc_savi_result = pc_savi_test.run_on_sequence(bernoulli_data[:, 0].astype(int), bernoulli_data[:, 1].astype(int))
    binary_nscore_result = binary_nscore_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    nscore_result = nscore_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])
    wsr_result = wsr_test.run_on_sequence(bernoulli_data[:, 0], bernoulli_data[:, 1])

    # Store the times-to-decision for each method
    times_to_decision_per_policy_close[i, 0] = step_result.info["Time"]
    times_to_decision_per_policy_close[i, 1] = savi_result.info["Time"]
    times_to_decision_per_policy_close[i, 2] = pc_savi_result.info["Time"]
    times_to_decision_per_policy_close[i, 3] = binary_nscore_result.info["Time"]
    times_to_decision_per_policy_close[i, 4] = nscore_result.info["Time"]
    times_to_decision_per_policy_close[i, 5] = wsr_result.info["Time"]

 20%|██        | 4/20 [00:49<03:34, 13.39s/it]/home/dsnyder5/miniconda3/envs/nscore/lib/python3.13/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
100%|██████████| 20/20 [03:39<00:00, 10.97s/it]


In [13]:
# Compute average times to decision for each method over n_reruns independent redraws
mean_time_to_decision_per_policy_close = np.mean(times_to_decision_per_policy_close, axis=0)

# Compute std of times to decision for each method over n_reruns independent redraws
std_time_to_decision_per_policy_close = np.std(times_to_decision_per_policy_close, axis=0)

print("(Small-Gap) Binary Metric Comparison Problem")
print()
print(f"True policy mean performance levels: Policy 0 = {policy_means_close[0]}, Policy 1 = {policy_means_close[1]}")
print()
print(f"Average (over {n_reruns} redraws) times to decision for each method (alpha = {alpha}): ")
print()
print(f"STEP: {mean_time_to_decision_per_policy_close[0]} ({std_time_to_decision_per_policy_close[0] / np.sqrt(n_reruns)})")
print(f"SAVI: {mean_time_to_decision_per_policy_close[1]} ({std_time_to_decision_per_policy_close[1] / np.sqrt(n_reruns)})")
print(f"Theta-SAVI: {mean_time_to_decision_per_policy_close[2]} ({std_time_to_decision_per_policy_close[2] / np.sqrt(n_reruns)})")
print(f"Binary NSCORE: {mean_time_to_decision_per_policy_close[3]} ({std_time_to_decision_per_policy_close[3] / np.sqrt(n_reruns)})")
print(f"NSCORE: {mean_time_to_decision_per_policy_close[4]} ({std_time_to_decision_per_policy_close[4] / np.sqrt(n_reruns)})")
print(f"WSR: {mean_time_to_decision_per_policy_close[5]} ({std_time_to_decision_per_policy_close[5] / np.sqrt(n_reruns)})")

(Small-Gap) Binary Metric Comparison Problem

True policy mean performance levels: Policy 0 = 0.45, Policy 1 = 0.55

Average (over 20 redraws) times to decision for each method (alpha = 0.05): 

STEP: 299.6 (33.64865524801845)
SAVI: 370.9 (37.36220684060297)
Theta-SAVI: 352.9 (36.07248951763657)
Binary NSCORE: 371.4 (38.74218630898364)
NSCORE: 355.15 (37.83862279470541)
WSR: 450.75 (33.03807462610374)
